## 1. Setup
Connect Colab to the BigQuery project so the notebook queries the same tested dbt marts used across the project.

In [1]:
from google.colab import auth

auth.authenticate_user()
print("Authenticated")


Authenticated


In [2]:
from google.cloud import bigquery
client = bigquery.Client(project="vantra-514719")

In [3]:
import pandas as pd
import numpy as np

In [4]:
sql = """
select * from dbt_bpan.fct_orders
where refund_date <= '2021-08-07'
  and region is not null;
"""

customers_df = client.query(sql).to_dataframe()
customers_df.head()

,order_id,user_id,purchase_date,ship_date,delivery_date,refund_date,account_created_date,product_name,usd_price,local_price,...,marketing_channel,brand,account_creation_method,country_code,region,is_zero_price,is_missing_price,is_refunded,is_loyalty_member,is_ship_before_purchase
0,c6424295b772165,81749050,2020-05-02,2020-05-04,2020-05-11,2020-05-28,2019-01-07,Thinkpad Laptop,1018.300000000,1018.300000000,...,direct,Lenovo,desktop,US,NA,False,False,True,False,False
1,1a36ace949c58358,8973eadc,2019-08-19,2019-08-20,2019-08-27,2019-09-01,2019-08-16,Thinkpad Laptop,1077.920000000,9998.000000000,...,direct,Lenovo,desktop,NO,EMEA,False,False,True,False,False
2,13dceb8394130560,43dbcafc,2020-07-03,2020-07-06,2020-07-08,2020-07-25,2020-06-23,Thinkpad Laptop,1091.900000000,1091.900000000,...,affiliate,Lenovo,unknown,US,NA,False,False,True,False,False
3,e3c9d284f6413786,1a83b8e1,2021-03-14,2021-03-15,2021-03-21,2021-04-11,2021-01-09,Samsung Charging Cable Pack,19.060000000,13.980000000,...,direct,Samsung,desktop,GB,EMEA,False,False,True,False,False
4,f57ac2d415710109,115780a5,2020-10-06,2020-10-09,2020-10-13,2020-10-14,2020-07-29,Macbook Air Laptop,502.840000000,3840.000000000,...,direct,Apple,desktop,TR,EMEA,False,False,True,False,False


In [5]:
customers_df['purchase_year'] = pd.to_datetime(customers_df['purchase_date']).dt.year
customers_df.head()

,order_id,user_id,purchase_date,ship_date,delivery_date,refund_date,account_created_date,product_name,usd_price,local_price,...,brand,account_creation_method,country_code,region,is_zero_price,is_missing_price,is_refunded,is_loyalty_member,is_ship_before_purchase,purchase_year
0,c6424295b772165,81749050,2020-05-02,2020-05-04,2020-05-11,2020-05-28,2019-01-07,Thinkpad Laptop,1018.300000000,1018.300000000,...,Lenovo,desktop,US,NA,False,False,True,False,False,2020
1,1a36ace949c58358,8973eadc,2019-08-19,2019-08-20,2019-08-27,2019-09-01,2019-08-16,Thinkpad Laptop,1077.920000000,9998.000000000,...,Lenovo,desktop,NO,EMEA,False,False,True,False,False,2019
2,13dceb8394130560,43dbcafc,2020-07-03,2020-07-06,2020-07-08,2020-07-25,2020-06-23,Thinkpad Laptop,1091.900000000,1091.900000000,...,Lenovo,unknown,US,NA,False,False,True,False,False,2020
3,e3c9d284f6413786,1a83b8e1,2021-03-14,2021-03-15,2021-03-21,2021-04-11,2021-01-09,Samsung Charging Cable Pack,19.060000000,13.980000000,...,Samsung,desktop,GB,EMEA,False,False,True,False,False,2021
4,f57ac2d415710109,115780a5,2020-10-06,2020-10-09,2020-10-13,2020-10-14,2020-07-29,Macbook Air Laptop,502.840000000,3840.000000000,...,Apple,desktop,TR,EMEA,False,False,True,False,False,2020


In [16]:
factors = ['product_name','brand','marketing_channel','purchase_platform','region','is_loyalty_member','purchase_year']

for items in factors:
  refund_rate = customers_df.groupby(items)['is_refunded'].size() /len(customers_df)
  print(refund_rate, "\n")


product_name
27in 4K Gaming Monitor         0.268815
Apple Airpods Headphones       0.490313
Apple Iphone                   0.004098
Macbook Air Laptop             0.084389
Samsung Charging Cable Pack    0.054396
Samsung Webcam                 0.034277
Thinkpad Laptop                0.063711
Name: is_refunded, dtype: Float64 

brand
Apple        0.5788
Generic    0.268815
Lenovo     0.063711
Samsung    0.088674
Name: is_refunded, dtype: Float64 

marketing_channel
affiliate       0.024776
direct           0.78614
email           0.164121
social media    0.018256
Name: is_refunded, dtype: Float64 

purchase_platform
mobile app    0.069858
website       0.930142
Name: is_refunded, dtype: Float64 

region
APAC     0.114195
EMEA      0.27161
LATAM    0.056073
NA       0.558122
Name: is_refunded, dtype: Float64 

is_loyalty_member
False    0.471684
True     0.528316
Name: is_refunded, dtype: Float64 

purchase_year
2019    0.179769
2020    0.579732
2021    0.240499
Name: is_refunded, dtype:

Decision

product, channel, platform, region and loyalty will be the features for this model. year might be significant but presents nothing meaningful for future predictions. brands are baked into the product.

